# 07 — PyTorch Comparison: Numerical Consistency

This notebook verifies that the JAX (flax.nnx) and PyTorch implementations produce **identical outputs and gradients** when given the same weights and inputs.

## Why this matters
- Confirms our JAX implementations are correct (not just that they converge)
- Lets you trust benchmark comparisons (notebook 08) — same computation, different runtime
- Serves as a migration sanity-check if porting an existing PyTorch model to JAX

## What we verify
1. Forward pass outputs match (tolerance < 1e-5)
2. Gradients w.r.t. a scalar loss match (tolerance < 1e-5)
3. Hidden state trajectories match step-by-step

In [ ]:
import sys; sys.path.insert(0, '..')
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import numpy as np
import torch
import torch.nn as tnn
import matplotlib.pyplot as plt

from src.rnn_jax import VanillaRNN, VanillaRNNModel, LowRankRNN, LowRankRNNModel
from src.rnn_torch import (
    VanillaRNN as TorchVanillaRNN,
    VanillaRNNModel as TorchVanillaRNNModel,
    LowRankRNN as TorchLowRankRNN,
    LowRankRNNModel as TorchLowRankRNNModel,
    load_vanilla_rnn_weights,
    load_lowrank_rnn_weights,
)

torch.manual_seed(0)
np.random.seed(0)
ATOL = 1e-5

## 7.1 Vanilla RNN: Forward Pass Consistency

In [ ]:
INPUT_SIZE, HIDDEN_SIZE, T_LEN, BATCH = 3, 16, 20, 4

# --- JAX model ---
jax_model = VanillaRNNModel(INPUT_SIZE, HIDDEN_SIZE, output_size=1, rngs=nnx.Rngs(0))

# Extract JAX weights
jax_kernel = np.array(jax_model.rnn.cell.linear.kernel[...])  # (hidden+in, hidden)
jax_bias   = np.array(jax_model.rnn.cell.linear.bias[...])    # (hidden,)
jax_ro_W   = np.array(jax_model.readout.linear.kernel[...])   # (hidden, out)
jax_ro_b   = np.array(jax_model.readout.linear.bias[...])     # (out,)

# --- PyTorch model with SAME weights ---
torch_model = TorchVanillaRNNModel(INPUT_SIZE, HIDDEN_SIZE, output_size=1)
load_vanilla_rnn_weights(torch_model.rnn, jax_kernel, jax_bias)
with torch.no_grad():
    torch_model.readout.weight.copy_(torch.tensor(jax_ro_W.T))
    torch_model.readout.bias.copy_(torch.tensor(jax_ro_b))

# --- Same random input — batch-first (B, T, I) ---
xs_np = np.random.randn(BATCH, T_LEN, INPUT_SIZE).astype(np.float32)
xs_jax   = jnp.array(xs_np)
xs_torch = torch.tensor(xs_np)

# --- Forward pass ---
out_jax   = np.array(jax_model(xs_jax))
out_torch = torch_model(xs_torch).detach().numpy()

max_diff = np.max(np.abs(out_jax - out_torch))
print(f'Vanilla RNN forward pass — max |jax - torch|: {max_diff:.2e}  (tol={ATOL})')
print(f'PASS: {max_diff < ATOL}')

## 7.2 Vanilla RNN: Gradient Consistency

In [ ]:
# --- JAX gradient ---
def jax_loss(model):
    out = model(xs_jax)
    return jnp.mean(out ** 2)

_, grads_jax = nnx.value_and_grad(jax_loss)(jax_model)
# grads_jax is an nnx Module with grad values at each param
jax_grad_kernel = np.array(grads_jax.rnn.cell.linear.kernel[...])
jax_grad_bias   = np.array(grads_jax.rnn.cell.linear.bias[...])

# --- PyTorch gradient ---
torch_model.zero_grad()
torch_out  = torch_model(xs_torch)
torch_loss = torch.mean(torch_out ** 2)
torch_loss.backward()
torch_grad_W = torch_model.rnn.cell.linear.weight.grad.numpy().T   # (hidden+in, hidden)
torch_grad_b = torch_model.rnn.cell.linear.bias.grad.numpy()

diff_kernel = np.max(np.abs(jax_grad_kernel - torch_grad_W))
diff_bias   = np.max(np.abs(jax_grad_bias   - torch_grad_b))
print(f'Gradient diff — kernel: {diff_kernel:.2e}   bias: {diff_bias:.2e}   (tol={ATOL})')
print(f'PASS: {diff_kernel < ATOL and diff_bias < ATOL}')

## 7.3 Low-Rank RNN: Consistency Check

In [ ]:
RANK = 4
jax_lr  = LowRankRNNModel(INPUT_SIZE, HIDDEN_SIZE, rank=RANK, output_size=1, rngs=nnx.Rngs(0))
torch_lr = TorchLowRankRNNModel(INPUT_SIZE, HIDDEN_SIZE, rank=RANK, output_size=1)

# Load weights from JAX into PyTorch
U_np    = np.array(jax_lr.rnn.cell.U[...])
V_np    = np.array(jax_lr.rnn.cell.V[...])
Win_k   = np.array(jax_lr.rnn.cell.W_in.kernel[...])
Win_b   = np.array(jax_lr.rnn.cell.W_in.bias[...])
ro_W_np = np.array(jax_lr.readout.linear.kernel[...])
ro_b_np = np.array(jax_lr.readout.linear.bias[...])

load_lowrank_rnn_weights(torch_lr.rnn, U_np, V_np, Win_k, Win_b)
with torch.no_grad():
    torch_lr.readout.weight.copy_(torch.tensor(ro_W_np.T))
    torch_lr.readout.bias.copy_(torch.tensor(ro_b_np))

# Forward
out_lr_jax   = np.array(jax_lr(xs_jax))
out_lr_torch = torch_lr(xs_torch).detach().numpy()
diff_fwd = np.max(np.abs(out_lr_jax - out_lr_torch))
print(f'Low-rank RNN forward pass — max diff: {diff_fwd:.2e}   PASS: {diff_fwd < ATOL}')

# Gradient
def jax_lr_loss(model):
    return jnp.mean(model(xs_jax) ** 2)

_, grads_lr = nnx.value_and_grad(jax_lr_loss)(jax_lr)
jax_grad_U = np.array(grads_lr.rnn.cell.U[...])

torch_lr.zero_grad()
torch_lr(xs_torch).pow(2).mean().backward()
torch_grad_U = torch_lr.rnn.cell.U.grad.numpy()

diff_U = np.max(np.abs(jax_grad_U - torch_grad_U))
print(f'Low-rank RNN grad U     — max diff: {diff_U:.2e}   PASS: {diff_U < ATOL}')

## 7.4 Hidden State Trajectory Comparison

In [ ]:
# Get hidden states at every timestep — both now return (B, T, H)
jax_hidden, _ = jax_model.rnn(xs_jax)         # (B, T, H)
torch_hidden, _ = torch_model.rnn(xs_torch)    # (B, T, H)

jax_h_np   = np.array(jax_hidden)
torch_h_np = torch_hidden.detach().numpy()

# Max diff over (B, H) at each timestep → shape (T,)
step_max_diffs = np.max(np.abs(jax_h_np - torch_h_np), axis=(0, 2))

plt.figure(figsize=(9, 3.5))
plt.semilogy(step_max_diffs, 'o-', ms=4)
plt.axhline(ATOL, color='red', linestyle='--', label=f'tol={ATOL}')
plt.xlabel('Timestep'); plt.ylabel('Max |h_jax - h_torch|')
plt.title('Hidden state difference per timestep (Vanilla RNN)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f'All timesteps within tolerance: {np.all(step_max_diffs < ATOL)}')

## 7.5 Why Small Differences Exist

Floating-point arithmetic is **not associative** — JAX (XLA) and PyTorch may fuse operations differently, reorder additions, or use different BLAS implementations. With float32, differences of order 1e-7 are expected even for identical algorithms.

In [ ]:
# Demonstrate: same operation, different order → different float32 result
a = np.random.randn(1000).astype(np.float32)
sum_forward  = np.float32(np.sum(a))                 # default accumulation order
sum_reversed = np.float32(np.sum(a[::-1]))           # reversed order
print(f'Sum (forward):  {sum_forward}')
print(f'Sum (reversed): {sum_reversed}')
print(f'Difference due to FP non-associativity: {abs(sum_forward - sum_reversed):.2e}')